# First Complete Creative Video — Colab (Full Pipeline)

This notebook runs the full vertical-slice pipeline in an isolated Colab T4 runtime: planning → visuals → TTS → music → montage.

IMPORTANT:
- Run on a fresh Colab GPU (T4) runtime.
- Runtime type: GPU. Hardware accelerator: GPU.
- This will create isolated venvs and may download model weights (large).
- Default TTS: qwen. To use Chatterbox, set TTS='chatterbox' in the config cell.


In [ ]:
# Configuration
BRANCH = 'tga892338-rgb-refactor-colab-ready'
REPO_URL = 'https://github.com/tga892338-rgb/openmontage-colab.git'
PROJECT_DIR = 'openmontage-colab'
TTS_BACKEND = 'qwen'  # or 'chatterbox'
# Use this to skip heavy installs during debug: set DRY_RUN = True
DRY_RUN = False
print('Configured TTS:', TTS_BACKEND)

In [ ]:
# Clone the repo and checkout the branch
import os, subprocess, sys
if not os.path.isdir(PROJECT_DIR):
    subprocess.check_call(['git','clone','-b',BRANCH,REPO_URL,PROJECT_DIR])
os.chdir(PROJECT_DIR)
print('CWD:', os.getcwd())
print('HEAD:'); subprocess.check_call(['git','rev-parse','--abbrev-ref','HEAD'])


In [ ]:
# Quick runtime check (GPU/CUDA)
import torch, subprocess, json, sys
gpu_available = torch.cuda.is_available() if hasattr(torch,'cuda') else False
print('torch found:', 'yes' if 'torch' in sys.modules else 'no')
print('CUDA available:', gpu_available)
if gpu_available:
    try:
        import pynvml
        pynvml.nvmlInit()
        h = pynvml.nvmlDeviceGetHandleByIndex(0)
        name = pynvml.nvmlDeviceGetName(h).decode()
        mem = pynvml.nvmlDeviceGetMemoryInfo(h).total
        print('GPU:', name, 'VRAM(bytes):', mem)
    except Exception as e:
        print('nvml not available:', e)


In [ ]:
# Run the Colab orchestration script which creates isolated venvs and runs the pipeline.
import subprocess, os, shlex
if DRY_RUN:
    print('DRY_RUN: skipping heavy installs and generation')
else:
    cmd = [sys.executable, 'scripts/colab_run_full_pipeline.py', '--project', 'projects/first-creative-video', '--tts', TTS_BACKEND, '--shots-json', 'projects/first-creative-video/shot_plan.json']
    print('Running orchestration:', ' '.join(shlex.quote(p) for p in cmd))
    subprocess.check_call(cmd)


In [ ]:
# Display final video if produced
from IPython.display import Video, display
final = 'projects/first-creative-video/final.mp4'
import os
if os.path.exists(final):
    print('Final video:', final)
    display(Video(final, embed=True))
else:
    print('Final video not found at', final)
